In [ ]:
!pip3 install git+https://github.com/NetManAIOps/sktime.git

# Moving Average (MA) with sktime

## 1. Simple MA (NaiveForecaster mean strategy)

In [ ]:
import matplotlib.pyplot as plt
from sktime.datasets import load_airline
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.naive import NaiveForecaster
from sktime.performance_metrics.forecasting import mean_absolute_error

y = load_airline()
split_point = int(len(y) * 0.8)
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]
fh = ForecastingHorizon(y_test.index, is_relative=False)

print(f"Train length: {len(y_train)}, Test length: {len(y_test)}")
y.head()

## 2. Exponential MA (Simple Exponential Smoothing)

In [ ]:
ma_window = 5
ma_forecaster = NaiveForecaster(strategy="mean", window_length=ma_window)
ma_forecaster.fit(y_train)
y_pred_ma = ma_forecaster.predict(fh)
mae_ma = mean_absolute_error(y_test, y_pred_ma)

plt.figure(figsize=(12, 6))
plt.plot(y_train.index, y_train.values, label="Train")
plt.plot(y_test.index, y_test.values, label="Test")
plt.plot(y_pred_ma.index, y_pred_ma.values, label=f"MA(w={ma_window})")
plt.title(f"Simple MA via sktime NaiveForecaster, MAE={mae_ma:.2f}")
plt.legend()
plt.grid(True)
plt.show()

## 3. Double EMA (Holt Linear Trend)

In [ ]:
from sktime.forecasting.exp_smoothing import ExponentialSmoothing

holt = ExponentialSmoothing(trend="add", seasonal=None)
holt.fit(y_train)
y_pred_holt = holt.predict(fh)
mae_holt = mean_absolute_error(y_test, y_pred_holt)

plt.figure(figsize=(12, 6))
plt.plot(y_train.index, y_train.values, label="Train")
plt.plot(y_test.index, y_test.values, label="Test")
plt.plot(y_pred_holt.index, y_pred_holt.values, label="Double EMA (Holt)")
plt.title(f"Double EMA (Holt) in sktime, MAE={mae_holt:.2f}")
plt.legend()
plt.grid(True)
plt.show()

## 4. Holt-Winters (Triple Exponential Smoothing)

In [ ]:
from sktime.forecasting.theta import ThetaForecaster

hw = ExponentialSmoothing(trend="add", seasonal="mul", sp=12)
hw.fit(y_train)
y_pred_hw = hw.predict(fh)
mae_hw = mean_absolute_error(y_test, y_pred_hw)

theta = ThetaForecaster(sp=12)
theta.fit(y_train)
y_pred_theta = theta.predict(fh)
mae_theta = mean_absolute_error(y_test, y_pred_theta)

plt.figure(figsize=(12, 6))
plt.plot(y_train.index, y_train.values, label="Train")
plt.plot(y_test.index, y_test.values, label="Test")
plt.plot(y_pred_hw.index, y_pred_hw.values, label="Holt-Winters")
plt.plot(y_pred_theta.index, y_pred_theta.values, label="Theta")
plt.title("Holt-Winters / Theta in sktime")
plt.legend()
plt.grid(True)
plt.show()

print(f"MAE - MA: {mae_ma:.2f}")
print(f"MAE - Double EMA (Holt): {mae_holt:.2f}")
print(f"MAE - Holt-Winters: {mae_hw:.2f}")
print(f"MAE - Theta: {mae_theta:.2f}")